In [ ]:
# =============================================================================
# ROBERTA-BASE + TOPIC FUSION — COLAB VERSION
# Baseline: Test Macro-F1: 0.6870 | Val Macro-F1: 0.6951
#
# What's new vs vanilla RoBERTa:
#   1. FusionDataset — caches topic_probs in VRAM alongside input_ids
#   2. RoBERTaFusionClassifier — two-stream architecture:
#        [CLS] 768 → Linear(768,256) → LayerNorm → GELU → Dropout(0.3)
#        topic   30 → Linear(30, 64)  → LayerNorm → GELU → Dropout(0.2)
#        → Concat(320) → Linear(320,12)
#   3. Epoch-1 freeze: RoBERTa body frozen, only fusion head trains.
#      Prevents pretrained gradients from drowning the randomly-init topic head.
#   4. Differential LR: RoBERTa body=1e-5, fusion head=5e-5.
#   5. Everything else identical to baseline for clean comparison:
#      batch=64, cosine scheduler, FP16, per-genre threshold sweep 0.20→0.85,
#      gradient checkpointing ON, patience=5.
#
# Files required in DATA dir (same folder as pkl files):
#   - data_splits.pkl          (existing)
#   - mlb.pkl                  (existing)
#   - topic_probs_train.npy    ← UPLOAD THIS
#   - topic_probs_val.npy      ← UPLOAD THIS
#   - topic_probs_test.npy     ← UPLOAD THIS
#
# Estimated runtime: ~3.5 hours (slightly longer due to freeze/unfreeze logic)
# =============================================================================




from google.colab import drive
drive.mount('/content/drive')

!pip uninstall -y scikit-learn
!pip install -q -U pandas transformers>=4.48.0 accelerate>=1.2.0 evaluate \
    datasets>=3.1.0 iterative-stratification rapidfuzz scikit-learn>=1.8.0

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — ENABLE GPU'}")
if torch.cuda.is_available():
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


import pickle, numpy as np, os, time
from datetime import timedelta

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark   = True

DATA = "/content/drive/MyDrive/nlp-genre-project-data/"
SAVE = "/content/"

with open(f"{DATA}data_splits.pkl", "rb") as f:
    splits = pickle.load(f)

X_train, y_train = splits["X_train"], splits["y_train"]
X_val,   y_val   = splits["X_val"],   splits["y_val"]
X_test,  y_test  = splits["X_test"],  splits["y_test"]

with open(f"{DATA}mlb.pkl", "rb") as f:
    mlb = pickle.load(f)

TARGET_GENRES = list(mlb.classes_)
NUM_LABELS    = len(TARGET_GENRES)   # 12
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tp_train = np.load(f"{DATA}topic_probs_train.npy").astype(np.float32)
tp_val   = np.load(f"{DATA}topic_probs_val.npy").astype(np.float32)
tp_test  = np.load(f"{DATA}topic_probs_test.npy").astype(np.float32)

TOPIC_DIM = tp_train.shape[1]   # 30

assert len(tp_train) == len(X_train), "topic_probs_train row count mismatch"
assert len(tp_val)   == len(X_val),   "topic_probs_val row count mismatch"
assert len(tp_test)  == len(X_test),  "topic_probs_test row count mismatch"

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Labels: {NUM_LABELS} | Topic dim: {TOPIC_DIM} | Device: {device}")


import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, hamming_loss, classification_report

def fmt(seconds):
    return str(timedelta(seconds=int(seconds)))

def eta(elapsed_steps, total_steps, elapsed_seconds):
    if elapsed_steps == 0:
        return "calculating..."
    return fmt((elapsed_seconds / elapsed_steps) * (total_steps - elapsed_steps))

def get_pos_weight(y_train, device):
    counts = y_train.sum(axis=0)
    weight = (len(y_train) - counts) / (counts + 1e-8)
    return torch.FloatTensor(weight).to(device)

def tune_threshold_per_genre(probs, labels, num_labels, target_genres):
    best_thresholds = np.zeros(num_labels)
    print("  Per-genre threshold tuning:")
    for i in range(num_labels):
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.20, 0.85, 0.05):
            preds = (probs[:, i] > t).astype(int)
            f1    = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds[i] = best_t
        print(f"    {target_genres[i]:<20} best_t={best_t:.2f}  F1={best_f1:.4f}")
    return best_thresholds

def evaluate_fusion(model, loader, thresholds, model_name, device):
    """Evaluation loop — handles the topic_probs key in batch."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"],
                           batch["attention_mask"],
                           batch["topic_probs"])
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs > thresholds).astype(int)
            all_preds.append(preds)
            all_labels.append(batch["labels"].cpu().numpy())
    P, L = np.vstack(all_preds), np.vstack(all_labels)
    print(f"\n=== {model_name} Test Results ===")
    print(f"Micro-F1:     {f1_score(L, P, average='micro',  zero_division=0):.4f}")
    print(f"Macro-F1:     {f1_score(L, P, average='macro',  zero_division=0):.4f}")
    print(f"Hamming Loss: {hamming_loss(L, P):.4f}")
    print(classification_report(L, P, target_names=TARGET_GENRES, zero_division=0))


MODEL_NAME = "roberta-base"


class FusionGenreDataset(Dataset):
    """
    Extends UltraFastGenreDataset by also caching topic_probs in VRAM.
    All three tensors (input_ids, attention_mask, topic_probs) live in VRAM —
    zero CPU↔GPU transfer per batch, same as baseline.
    """
    def __init__(self, texts, labels, topic_probs, tokenizer,
                 max_len=512, device="cuda"):
        self.labels      = torch.FloatTensor(labels).to(device)
        self.topic_probs = torch.FloatTensor(topic_probs).to(device)   # (N, 30)

        print(f"Tokenizing {len(texts)} texts → VRAM...")
        enc = tokenizer(
            list(texts),
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        self.input_ids      = enc["input_ids"].to(device)
        self.attention_mask = enc["attention_mask"].to(device)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "topic_probs":    self.topic_probs[idx],
            "labels":         self.labels[idx]
        }


class RoBERTaFusionClassifier(nn.Module):
    """
    Two-stream architecture:

    Stream A — RoBERTa [CLS]:
        roberta(input_ids, mask) → cls (768)
        → Linear(768, 256) → LayerNorm(256) → GELU → Dropout(0.3)
        → cls_proj (256)

    Stream B — Topic probs:
        topic_probs (30)
        → Linear(30, 64) → LayerNorm(64) → GELU → Dropout(0.2)
        → topic_proj (64)

    Fusion:
        Concat([cls_proj, topic_proj]) (320)
        → Linear(320, 12)

    Design rationale:
    - Separate LayerNorm per stream normalizes scale mismatch before concat.
      CLS activations are ~N(0,1); topic probs are in [0,1] with mean ~0.03.
      Without LN the linear fusion layer would see a 30× scale difference.
    - GELU (not ReLU) — matches RoBERTa's internal activation, smoother
      gradient flow for small topic head early in training.
    - Dropout on each stream independently — allows model to learn to rely
      on whichever stream is more reliable for each genre.
    """
    def __init__(self, topic_dim, num_labels):
        super().__init__()

        # Stream A — RoBERTa encoder
        self.roberta = AutoModel.from_pretrained(MODEL_NAME)
        self.roberta.gradient_checkpointing_enable()

        self.cls_proj = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # Stream B — Topic projection
        self.topic_proj = nn.Sequential(
            nn.Linear(topic_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        # Fusion head
        self.classifier = nn.Linear(256 + 64, num_labels)

        # Weight init for new layers (RoBERTa body uses pretrained weights)
        self._init_weights()

    def _init_weights(self):
        for module in [self.cls_proj, self.topic_proj, self.classifier]:
            for layer in (module if isinstance(module, nn.Sequential) else [module]):
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    nn.init.zeros_(layer.bias)

    def forward(self, input_ids, attention_mask, topic_probs):
        # Stream A
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # (B, 768)
        cls_out = self.cls_proj(cls)            # (B, 256)

        # Stream B
        topic_out = self.topic_proj(topic_probs)  # (B, 64)

        # Fuse
        fused = torch.cat([cls_out, topic_out], dim=1)   # (B, 320)
        return self.classifier(fused)                     # (B, 12)

    def freeze_roberta(self):
        """Freeze RoBERTa body — only fusion head trains (used in epoch 1)."""
        for param in self.roberta.parameters():
            param.requires_grad = False

    def unfreeze_roberta(self):
        """Unfreeze for full end-to-end fine-tuning from epoch 2 onward."""
        for param in self.roberta.parameters():
            param.requires_grad = True


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tok_start = time.time()
print("\n--- Loading data + topic probs directly into GPU VRAM ---")
train_ds = FusionGenreDataset(X_train, y_train, tp_train, tokenizer, device=device)
val_ds   = FusionGenreDataset(X_val,   y_val,   tp_val,   tokenizer, device=device)
test_ds  = FusionGenreDataset(X_test,  y_test,  tp_test,  tokenizer, device=device)
print(f"All splits loaded in {fmt(time.time() - tok_start)}")

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM after data load: {used:.1f} GB / {total:.1f} GB ({100*used/total:.1f}% used)")

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


model     = RoBERTaFusionClassifier(TOPIC_DIM, NUM_LABELS).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))

used_after_model = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after model load: {used_after_model:.1f} GB / {total:.1f} GB ({100*used_after_model/total:.1f}% used)")

roberta_params = list(model.roberta.parameters())
fusion_params  = (list(model.cls_proj.parameters()) +
                  list(model.topic_proj.parameters()) +
                  list(model.classifier.parameters()))

optimizer = AdamW([
    {"params": roberta_params, "lr": 1e-5},
    {"params": fusion_params,  "lr": 5e-5},
], weight_decay=0.01)

EPOCHS      = 8
total_steps = len(train_loader) * EPOCHS   # scheduler counts all steps

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_macro            = 0.0
best_thresholds_saved = np.full(NUM_LABELS, 0.5)
patience              = 5
no_improve            = 0
train_start           = time.time()
global_step           = 0

print("=" * 65)
print(f"RoBERTa-Fusion training started.")
print(f"Total steps: {total_steps} | Steps/epoch: {len(train_loader)}")
print(f"Batch size: 64 | Max len: 512 | Precision: FP16 | Topic dim: {TOPIC_DIM}")
print(f"Epochs: {EPOCHS} | Patience: {patience} | Scheduler: Cosine")
print(f"LR — RoBERTa body: 1e-5 | Fusion head: 5e-5")
print(f"Epoch 1: RoBERTa FROZEN (fusion head warms up)")
print(f"Epoch 2+: Full end-to-end fine-tuning")
print(f"Baseline to beat → Val: 0.6951 | Test: 0.6870")
print("=" * 65)

model.freeze_roberta()
print("\n[Epoch 1] RoBERTa body FROZEN — training fusion head only")

for epoch in range(EPOCHS):
    epoch_start  = time.time()
    running_loss = 0.0
    model.train()

    # Unfreeze from epoch 2 onward
    if epoch == 1:
        model.unfreeze_roberta()
        print(f"\n[Epoch {epoch+1}] RoBERTa body UNFROZEN — full fine-tuning begins")

    print(f"\n{'─'*65}")
    print(f"EPOCH {epoch+1}/{EPOCHS}  |  Elapsed: {fmt(time.time() - train_start)}")
    frozen_status = "FROZEN" if epoch == 0 else "UNFROZEN"
    print(f"RoBERTa body: {frozen_status}")
    print(f"{'─'*65}")

    for step, batch in enumerate(train_loader):
        step_start = time.time()
        optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(batch["input_ids"],
                           batch["attention_mask"],
                           batch["topic_probs"])
            loss   = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= scale_before:
            scheduler.step()

        running_loss += loss.item()
        global_step  += 1

        if (step + 1) % 100 == 0:
            avg_loss      = running_loss / (step + 1)
            epoch_elapsed = time.time() - epoch_start
            total_elapsed = time.time() - train_start
            epoch_eta_s   = (epoch_elapsed / (step + 1)) * (len(train_loader) - step - 1)
            print(
                f"  Step {step+1:>4}/{len(train_loader)}"
                f"  |  Loss: {avg_loss:.4f}"
                f"  |  Step: {time.time()-step_start:.2f}s"
                f"  |  Epoch ETA: {fmt(epoch_eta_s)}"
                f"  |  Total ETA: {eta(global_step, total_steps, total_elapsed)}"
            )

    print(f"\nEpoch {epoch+1} training done in {fmt(time.time() - epoch_start)}")

    # ── Validation ────────────────────────────────────────────────────────────
    val_start = time.time()
    model.eval()
    all_logits_v, all_labels_v = [], []

    with torch.no_grad():
        for b in val_loader:
            logits = model(b["input_ids"], b["attention_mask"], b["topic_probs"])
            all_logits_v.append(logits.cpu().numpy())
            all_labels_v.append(b["labels"].cpu().numpy())

    all_logits_np = np.vstack(all_logits_v)
    all_labels_np = np.vstack(all_labels_v)
    all_probs_np  = 1 / (1 + np.exp(-all_logits_np))

    best_t_epoch     = tune_threshold_per_genre(
        all_probs_np, all_labels_np, NUM_LABELS, TARGET_GENRES
    )
    best_preds_epoch = (all_probs_np > best_t_epoch).astype(int)
    ep_macro_f1      = f1_score(all_labels_np, best_preds_epoch,
                                average="macro", zero_division=0)

    total_elapsed = time.time() - train_start
    print(f"\nValidation done in {fmt(time.time() - val_start)}")
    print(f"Epoch {epoch+1}/{EPOCHS}  |  Val Macro-F1: {ep_macro_f1:.4f}  "
          f"(baseline: 0.6951, delta: {ep_macro_f1-0.6951:+.4f})")
    print(f"Total elapsed: {fmt(total_elapsed)}  |  "
          f"ETA: {eta(global_step, total_steps, total_elapsed)}")

    if ep_macro_f1 > best_macro:
        best_macro            = ep_macro_f1
        best_thresholds_saved = best_t_epoch.copy()
        no_improve            = 0
        torch.save(model.state_dict(), f"{SAVE}roberta_fusion_best.pt")
        print(f"  ->  New best! Saved roberta_fusion_best.pt")
        print(f"  -> Thresholds: {dict(zip(TARGET_GENRES, best_t_epoch.round(2)))}")
    else:
        no_improve += 1
        print(f"  -> No improvement ({no_improve}/{patience})")

    if no_improve >= patience:
        print("Early stopping triggered.")
        break

total_time = time.time() - train_start
print(f"\n{'='*65}")
print(f"Training complete. Total time: {fmt(total_time)}")
print(f"Best Val Macro-F1: {best_macro:.4f}  (baseline delta: {best_macro-0.6951:+.4f})")
print(f"{'='*65}")


eval_start = time.time()

print("\nLoading best model for final test evaluation...")
model.load_state_dict(torch.load(
    f"{SAVE}roberta_fusion_best.pt",
    map_location=device,
    weights_only=True
))

model.eval()
all_logits_v, all_labels_v = [], []
with torch.no_grad():
    for b in val_loader:
        logits = model(b["input_ids"], b["attention_mask"], b["topic_probs"])
        all_logits_v.append(logits.cpu().numpy())
        all_labels_v.append(b["labels"].cpu().numpy())

all_probs_val  = 1 / (1 + np.exp(-np.vstack(all_logits_v)))
all_labels_val = np.vstack(all_labels_v)

final_thresholds = tune_threshold_per_genre(
    all_probs_val, all_labels_val, NUM_LABELS, TARGET_GENRES
)

evaluate_fusion(model, test_loader, final_thresholds, "RoBERTa-Fusion", device)

print("\n── Per-genre delta vs baseline ──────────────────────────────────")
baseline_f1 = {
    "Action": 0.64, "Comedy": 0.69, "Drama": 0.75, "Horror": 0.71,
    "Thriller": 0.71, "Sci-Fi": 0.75, "Fantasy": 0.61, "Romance": 0.57,
    "Animation": 0.62, "Documentary": 0.84, "Literary Fiction": 0.64,
    "Non-Fiction": 0.70
}
print(f"  {'Genre':<20} {'Baseline':>8} {'Fusion':>8} {'Delta':>8}")
print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*8}")

model.eval()
all_preds_f, all_labels_f = [], []
with torch.no_grad():
    for b in test_loader:
        logits = model(b["input_ids"], b["attention_mask"], b["topic_probs"])
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs > final_thresholds).astype(int)
        all_preds_f.append(preds)
        all_labels_f.append(b["labels"].cpu().numpy())
P_f = np.vstack(all_preds_f); L_f = np.vstack(all_labels_f)

for i, genre in enumerate(TARGET_GENRES):
    fusion_f1   = f1_score(L_f[:, i], P_f[:, i], zero_division=0)
    base_f1     = baseline_f1.get(genre, 0.0)
    delta       = fusion_f1 - base_f1
    flag        = " " if delta > 0.005 else (" " if delta < -0.005 else "  ~")
    print(f"  {genre:<20} {base_f1:>8.4f} {fusion_f1:>8.4f} {delta:>+8.4f}{flag}")

print(f"\nEvaluation done in {fmt(time.time() - eval_start)}")
print(f"Total session time: {fmt(time.time() - train_start)}")


import shutil

source = f"{SAVE}roberta_fusion_best.pt"
backup = f"{DATA}roberta_fusion_best.pt"

print("\nStarting model backup...")
if os.path.exists(source):
    shutil.copy2(source, backup)
    print(f" Backed up to: {backup}")
else:
    print(f" Could not find {source}. Did training complete?")

Mounted at /content/drive
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.

PyTorch: 2.10.0+cu128
GPU:     Tesla T4
VRAM:    15.6 GB
Train: 36104 | Val: 7774 | Test: 7740
Labels: 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...


--- Loading data + topic probs directly into GPU VRAM ---
Tokenizing 36104 texts → VRAM...
Tokenizing 7774 texts → VRAM...
Tokenizing 7740 texts → VRAM...
All splits loaded in 0:00:48
VRAM after data load: 0.4 GB / 15.6 GB (2.8% used)
Train batches: 565 | Val: 122 | Test: 121


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


VRAM after model load: 0.9 GB / 15.6 GB (6.0% used)
RoBERTa-Fusion training started.
Total steps: 4520 | Steps/epoch: 565
Batch size: 64 | Max len: 512 | Precision: FP16 | Topic dim: 30
Epochs: 8 | Patience: 5 | Scheduler: Cosine
LR — RoBERTa body: 1e-5 | Fusion head: 5e-5
Epoch 1: RoBERTa FROZEN (fusion head warms up)
Epoch 2+: Full end-to-end fine-tuning
Baseline to beat → Val: 0.6951 | Test: 0.6870

[Epoch 1] RoBERTa body FROZEN — training fusion head only

─────────────────────────────────────────────────────────────────
EPOCH 1/8  |  Elapsed: 0:00:00
RoBERTa body: FROZEN
─────────────────────────────────────────────────────────────────
  Step  100/565  |  Loss: 1.4089  |  Step: 1.56s  |  Epoch ETA: 0:11:59  |  Total ETA: 1:53:59
  Step  200/565  |  Loss: 1.3808  |  Step: 1.57s  |  Epoch ETA: 0:09:27  |  Total ETA: 1:52:01
  Step  300/565  |  Loss: 1.3498  |  Step: 1.56s  |  Epoch ETA: 0:06:53  |  Total ETA: 1:49:37
  Step  400/565  |  Loss: 1.3261  |  Step: 1.56s  |  Epoch ETA: 0: